# PW01 · Source Event Shards

用途：在 Colab 冷启动环境中执行单个 sample_role 的 source shard。formal 主流程角色为 positive_source 与 clean_negative；planner_conditioned_control_negative 仅作为 optional diagnostic cohort 按需执行。

边界：仅执行 PW01，不改动主链语义，不扩展到后续 stage。

## 1) 定义路径与参数

本单元定义 PW01 的输入参数，并固定脚本入口路径。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "PW01_Source_Event_Shards"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
DRIVE_MOUNT_ROOT = Path("/content/drive")
LOCAL_RUNTIME_ENABLED = True
LOCAL_PROJECT_ROOT = Path("/content/CEG_WM_PaperWorkflow")
PERSISTENT_DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow"
ATTESTATION_PROJECT_ROOT = PERSISTENT_DRIVE_PROJECT_ROOT
DRIVE_BUNDLE_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow_Bundles"
if LOCAL_RUNTIME_ENABLED:
    DRIVE_PROJECT_ROOT = LOCAL_PROJECT_ROOT
else:
    DRIVE_PROJECT_ROOT = PERSISTENT_DRIVE_PROJECT_ROOT
REPO_ROOT = Path("/content/ceg_wm_workspace")
SCRIPT_PATH = REPO_ROOT / "paper_workflow" / "scripts" / "PW01_Source_Event_Shards.py"
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
DRIVE_MODELS_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "Models"
PERSISTENT_INSPYRENET_ROOT = DRIVE_MODELS_ROOT / "inspyrenet"
PERSISTENT_HF_ROOT = DRIVE_MODELS_ROOT / "Huggingface"
LOCAL_HF_HOME = REPO_ROOT / "huggingface_cache"
LOCAL_HF_HUB_CACHE = LOCAL_HF_HOME / "hub"
LOCAL_TRANSFORMERS_CACHE = LOCAL_HF_HOME / "transformers"
SEMANTIC_MODEL_SOURCE_URLS = {
    "inspyrenet": "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth",
}

FAMILY_ID = "paper_eval_family_geometry_mix_v2"
SAMPLE_ROLE = "positive_source"
SAMPLE_ROLE_DIRECTORY_NAMES = {
    "positive_source": "positive",
    "clean_negative": "negative",
    "planner_conditioned_control_negative": "control_negative",
}
if SAMPLE_ROLE not in SAMPLE_ROLE_DIRECTORY_NAMES:
    raise ValueError(f"unsupported SAMPLE_ROLE: {SAMPLE_ROLE}")
SAMPLE_ROLE_DIRNAME = SAMPLE_ROLE_DIRECTORY_NAMES[SAMPLE_ROLE]
SHARD_INDEX = 0             # 当前会话跑哪个 shard
SHARD_COUNT = 3             # 总 shard 数，默认与 PW00 一致
PW01_WORKER_COUNT = 1       # 单卡单路或单卡双路，当前只允许 1 或 2
FORCE_RERUN = False

def run_checked(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

## 2) 挂载 Drive 并同步仓库

本单元保证 PW01 可以在空白 Colab 会话中直接冷启动。

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
    DRIVE_MOUNT_STATUS = "mounted"
except Exception as exc:
    DRIVE_MOUNT_STATUS = f"skipped: {type(exc).__name__}: {exc}"

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HOME.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_TRANSFORMERS_CACHE.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    origin_url = origin_result.stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from paper_workflow.scripts.pw_local_runtime import prepare_local_runtime_for_stage
from scripts.notebook_runtime_common import resolve_notebook_model_cache_layout

if LOCAL_RUNTIME_ENABLED:
    prepare_local_runtime_for_stage(
        stage_name="PW01_Source_Event_Shards",
        family_id=FAMILY_ID,
        local_project_root=LOCAL_PROJECT_ROOT,
        drive_bundle_root=DRIVE_BUNDLE_ROOT,
        sample_role=SAMPLE_ROLE,
        shard_index=SHARD_INDEX,
        shard_count=SHARD_COUNT,
        clean_before_run=True,
        include_optional_control_negative=False,
    )

MODEL_CACHE_LAYOUT = resolve_notebook_model_cache_layout(DRIVE_MOUNT_ROOT, REPO_ROOT, create_directories=True)
DRIVE_MODELS_ROOT = MODEL_CACHE_LAYOUT["drive_models_root"]
PERSISTENT_INSPYRENET_ROOT = MODEL_CACHE_LAYOUT["persistent_inspyrenet_root"]
PERSISTENT_HF_ROOT = MODEL_CACHE_LAYOUT["persistent_hf_root"]
LOCAL_HF_HOME = MODEL_CACHE_LAYOUT["local_hf_home"]
LOCAL_HF_HUB_CACHE = MODEL_CACHE_LAYOUT["local_hf_hub_cache"]
LOCAL_TRANSFORMERS_CACHE = MODEL_CACHE_LAYOUT["local_transformers_cache"]

print_json(
    "repo_bootstrap",
    {
        "drive_mount_status": DRIVE_MOUNT_STATUS,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "script_path": str(SCRIPT_PATH),
        "drive_models_root": str(DRIVE_MODELS_ROOT),
        "persistent_inspyrenet_root": str(PERSISTENT_INSPYRENET_ROOT),
        "persistent_hf_root": str(PERSISTENT_HF_ROOT),
        "local_hf_home": str(LOCAL_HF_HOME),
        "model_cache_mode": "local_session_primary",
        "persistent_hf_root_role": "compatibility_only",
    },
)

## 3) 安装依赖并执行 attestation bootstrap

本单元在 paper_workflow 活动主路径内完成环境准备：安装 transparent-background、准备 InSPyReNet 权重、准备 Hugging Face snapshot，并完成 attestation env bootstrap。

In [ ]:
os.environ["HF_HOME"] = str(LOCAL_HF_HOME)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_TRANSFORMERS_CACHE)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if Path("/usr/bin/apt-get").exists():
    subprocess.run(["apt-get", "update", "-qq"], check=False, capture_output=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"],
        check=False,
        capture_output=True,
    )

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)

if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

run_checked([sys.executable, "-m", "pip", "install", "transparent-background"], cwd=REPO_ROOT)

from huggingface_hub import HfApi
from scripts.notebook_runtime_common import (
    bootstrap_notebook_model_cache,
    collect_weight_summary,
    ensure_attestation_env_bootstrap,
    load_yaml_mapping,
    resolve_model_identity,
)

CFG_OBJ = load_yaml_mapping(CONFIG_PATH)
MODEL_IDENTITY = resolve_model_identity(CFG_OBJ)
PROMPT_SOURCE_PATH = REPO_ROOT / str(CFG_OBJ.get("inference_prompt_file", ""))
FORCE_WEIGHT_DOWNLOAD = str(os.environ.get("INSPYRENET_FORCE_DOWNLOAD", "")).strip().lower() in {"1", "true", "yes"}
WEIGHT_URL_OVERRIDE = str(os.environ.get("INSPYRENET_MODEL_URL", "")).strip() or None

HF_ACCESS_SUMMARY = {"status": "unknown", "detail": "not_checked"}
try:
    hf_user = HfApi().whoami()
    HF_ACCESS_SUMMARY = {"status": "authenticated", "detail": hf_user.get("name", "unknown")}
except Exception:
    HF_ACCESS_SUMMARY = {"status": "anonymous", "detail": "anonymous access"}

MODEL_CACHE_BOOTSTRAP = bootstrap_notebook_model_cache(
    CFG_OBJ,
    DRIVE_MOUNT_ROOT,
    REPO_ROOT,
    semantic_model_source_urls=SEMANTIC_MODEL_SOURCE_URLS,
    weight_url_override=WEIGHT_URL_OVERRIDE,
    force_inspyrenet_download=FORCE_WEIGHT_DOWNLOAD,
)
MODEL_SNAPSHOT_PATH = str(MODEL_CACHE_BOOTSTRAP["local_snapshot_path"])
WEIGHT_PATH = Path(str(MODEL_CACHE_BOOTSTRAP["repo_inspyrenet_path"]))
PERSISTENT_WEIGHT_PATH = Path(str(MODEL_CACHE_BOOTSTRAP["persistent_inspyrenet_path"]))
MODEL_DOWNLOAD_SUMMARY = dict(MODEL_CACHE_BOOTSTRAP["model_audit_summary"])
WEIGHT_DOWNLOAD_SUMMARY = collect_weight_summary(REPO_ROOT, CFG_OBJ)
CONFIG_BINDINGS = {
    "prompt_source_path": str(PROMPT_SOURCE_PATH),
    "prompt_exists": PROMPT_SOURCE_PATH.exists(),
    "model_id": MODEL_IDENTITY["model_id"],
    "hf_revision": MODEL_IDENTITY["revision"],
    "hf_access": HF_ACCESS_SUMMARY,
    "local_hf_home": MODEL_CACHE_BOOTSTRAP["local_hf_home"],
    "local_hf_hub_cache": MODEL_CACHE_BOOTSTRAP["local_hf_hub_cache"],
    "local_transformers_cache": MODEL_CACHE_BOOTSTRAP["local_transformers_cache"],
    "model_snapshot_path": MODEL_SNAPSHOT_PATH,
    "semantic_model_path": str(WEIGHT_PATH),
    "persistent_inspyrenet_path": str(PERSISTENT_WEIGHT_PATH),
    "semantic_model_source": MODEL_CACHE_BOOTSTRAP["semantic_model_source"],
    "semantic_model_url": MODEL_CACHE_BOOTSTRAP["semantic_model_url"],
    "force_weight_download": FORCE_WEIGHT_DOWNLOAD,
    "snapshot_source": MODEL_CACHE_BOOTSTRAP["snapshot_source"],
    "cache_reuse_mode": MODEL_CACHE_BOOTSTRAP["cache_reuse_mode"],
    "weight_cache_mode": MODEL_CACHE_BOOTSTRAP["weight_cache_mode"],
    "model_source_binding": MODEL_CACHE_BOOTSTRAP["model_source_binding"],
}
ATTESTATION_BOOTSTRAP = ensure_attestation_env_bootstrap(
    CFG_OBJ,
    ATTESTATION_PROJECT_ROOT,
    allow_generate=True,
    allow_missing=False,
)
print_json("model_cache_bootstrap", MODEL_CACHE_BOOTSTRAP)
print_json("config_bindings", CONFIG_BINDINGS)
print_json("model_download_summary", MODEL_DOWNLOAD_SUMMARY)
print_json("weight_download_summary", WEIGHT_DOWNLOAD_SUMMARY)
print_json("attestation_env_bootstrap", ATTESTATION_BOOTSTRAP)
run_checked(["nvidia-smi"])

## 4) 运行前 precheck

本单元同时检查 PW01 必要输入，以及 paper_workflow 活动主路径所需的 GPU / 模型 / attestation 运行前条件。

In [ ]:
import os
import shutil
import socket

import torch
from huggingface_hub import HfApi
from scripts.notebook_runtime_common import (
    apply_notebook_model_snapshot_binding,
    load_yaml_mapping,
    write_yaml_mapping,
)
from scripts.workflow_acceptance_common import detect_pw01_preflight

FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
FAMILY_MANIFEST_PATH = FAMILY_ROOT / "manifests" / "paper_eval_family_manifest.json"
SOURCE_EVENT_GRID_PATH = FAMILY_ROOT / "manifests" / "source_event_grid.jsonl"
SOURCE_SHARD_PLAN_PATH = FAMILY_ROOT / "manifests" / "source_shard_plan.json"
SOURCE_SPLIT_PLAN_PATH = FAMILY_ROOT / "manifests" / "source_split_plan.json"
PROJECT_ROOT_PRECHECK_LABEL = "项目运行根目录存在" if LOCAL_RUNTIME_ENABLED else "Drive 项目根目录存在"

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck(PROJECT_ROOT_PRECHECK_LABEL, DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
record_precheck("attestation 持久根目录存在", ATTESTATION_PROJECT_ROOT.exists(), str(ATTESTATION_PROJECT_ROOT))
record_precheck("Drive 模型根目录存在", DRIVE_MODELS_ROOT.exists(), str(DRIVE_MODELS_ROOT))
record_precheck("persistent InSPyReNet 目录存在", PERSISTENT_INSPYRENET_ROOT.exists(), str(PERSISTENT_INSPYRENET_ROOT))
record_precheck("persistent Huggingface 路径仅兼容保留", True, str(PERSISTENT_HF_ROOT))
record_precheck("本地 HF_HOME 目录存在", LOCAL_HF_HOME.exists(), str(LOCAL_HF_HOME))
record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
record_precheck("PW01 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck("family 根目录存在", FAMILY_ROOT.exists(), str(FAMILY_ROOT))
record_precheck("family manifest 存在", FAMILY_MANIFEST_PATH.exists(), str(FAMILY_MANIFEST_PATH))
record_precheck("source event grid 存在", SOURCE_EVENT_GRID_PATH.exists(), str(SOURCE_EVENT_GRID_PATH))
record_precheck("source shard plan 存在", SOURCE_SHARD_PLAN_PATH.exists(), str(SOURCE_SHARD_PLAN_PATH))
record_precheck("source split plan 存在", SOURCE_SPLIT_PLAN_PATH.exists(), str(SOURCE_SPLIT_PLAN_PATH))
record_precheck(
    "sample_role 合法",
    SAMPLE_ROLE in SAMPLE_ROLE_DIRECTORY_NAMES,
    f"sample_role={SAMPLE_ROLE}",
)
record_precheck(
    "shard 参数合法",
    isinstance(SHARD_COUNT, int) and SHARD_COUNT > 0 and isinstance(SHARD_INDEX, int) and 0 <= SHARD_INDEX < SHARD_COUNT,
    f"shard_index={SHARD_INDEX}, shard_count={SHARD_COUNT}",
)
record_precheck(
    "PW01_WORKER_COUNT 合法",
    isinstance(PW01_WORKER_COUNT, int) and PW01_WORKER_COUNT in {1, 2},
    f"pw01_worker_count={PW01_WORKER_COUNT}",
)
record_precheck(
    "MODEL_SNAPSHOT_PATH 已提供",
    isinstance(MODEL_SNAPSHOT_PATH, str) and bool(str(MODEL_SNAPSHOT_PATH).strip()),
    str(MODEL_SNAPSHOT_PATH),
)
record_precheck(
    "MODEL_SNAPSHOT_PATH 目录存在",
    Path(str(MODEL_SNAPSHOT_PATH)).exists() and Path(str(MODEL_SNAPSHOT_PATH)).is_dir(),
    str(MODEL_SNAPSHOT_PATH),
)
record_precheck(
    "模型 snapshot 来源为本地会话缓存",
    str(MODEL_CACHE_BOOTSTRAP.get("snapshot_source", "")) in {"local_session_cache", "downloaded_this_session"},
    json.dumps({"snapshot_source": MODEL_CACHE_BOOTSTRAP.get("snapshot_source", "<absent>"), "cache_reuse_mode": MODEL_CACHE_BOOTSTRAP.get("cache_reuse_mode", "<absent>")}, ensure_ascii=False, sort_keys=True),
)
record_precheck(
    "MODEL_SNAPSHOT_PATH 位于本地 HF_HOME",
    str(Path(str(MODEL_SNAPSHOT_PATH)).resolve()).startswith(str(LOCAL_HF_HOME.resolve())),
    f"model_snapshot_path={MODEL_SNAPSHOT_PATH}, local_hf_home={LOCAL_HF_HOME}",
)
record_precheck(
    "repo InSPyReNet 权重存在",
    WEIGHT_PATH.exists() and WEIGHT_PATH.is_file(),
    str(WEIGHT_PATH),
)

family_manifest_obj = {}
manifest_shard_count = None
manifest_sample_roles = []
if FAMILY_MANIFEST_PATH.exists():
    family_manifest_obj = json.loads(FAMILY_MANIFEST_PATH.read_text(encoding="utf-8"))
    source_parameters = family_manifest_obj.get("source_parameters", {})
    if isinstance(source_parameters, dict):
        manifest_shard_count = source_parameters.get("source_shard_count")
    sample_roles_node = family_manifest_obj.get("sample_roles", {})
    if isinstance(sample_roles_node, dict):
        active_roles = sample_roles_node.get("active")
        if isinstance(active_roles, list):
            manifest_sample_roles = [str(item) for item in active_roles if isinstance(item, str)]

record_precheck(
    "manifest 与 SHARD_COUNT 一致",
    isinstance(manifest_shard_count, int) and manifest_shard_count == SHARD_COUNT,
    f"manifest_shard_count={manifest_shard_count}, shard_count={SHARD_COUNT}",
)
record_precheck(
    "manifest 激活当前 sample_role",
    SAMPLE_ROLE in manifest_sample_roles,
    json.dumps(manifest_sample_roles, ensure_ascii=False),
)
record_precheck("FORCE_RERUN 为 bool", isinstance(FORCE_RERUN, bool), str(FORCE_RERUN))
record_precheck(
    "attestation bootstrap 状态可用",
    str(ATTESTATION_BOOTSTRAP.get("status", "")) in {"generated", "reused", "disabled"},
    json.dumps(ATTESTATION_BOOTSTRAP, ensure_ascii=False, sort_keys=True),
)

PRECHECK_RUNTIME_METADATA_ROOT = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME / "precheck_runtime_metadata"
PRECHECK_BOUND_CONFIG_PATH = PRECHECK_RUNTIME_METADATA_ROOT / "precheck_bound_config.yaml"
PW01_BOUND_CONFIG_PATH = PRECHECK_BOUND_CONFIG_PATH
PRECHECK_BINDING_ENV = dict(os.environ)
PRECHECK_BINDING_ENV["CEG_WM_MODEL_SNAPSHOT_PATH"] = str(MODEL_SNAPSHOT_PATH)
PRECHECK_BASE_CFG = load_yaml_mapping(CONFIG_PATH)
PRECHECK_BOUND_CFG = apply_notebook_model_snapshot_binding(
    PRECHECK_BASE_CFG,
    env_mapping=PRECHECK_BINDING_ENV,
)
write_yaml_mapping(PW01_BOUND_CONFIG_PATH, PRECHECK_BOUND_CFG)
PRECHECK_MODEL_SOURCE_BINDING = (
    PRECHECK_BOUND_CFG.get("model_source_binding")
    if isinstance(PRECHECK_BOUND_CFG.get("model_source_binding"), dict)
    else None
)
PRECHECK_BINDING_APPLIED = (
    PRECHECK_MODEL_SOURCE_BINDING is not None
    and isinstance(PRECHECK_BOUND_CFG.get("model_snapshot_path"), str)
    and bool(str(PRECHECK_BOUND_CFG.get("model_snapshot_path")).strip())
)
if PRECHECK_MODEL_SOURCE_BINDING is not None:
    MODEL_DOWNLOAD_SUMMARY["binding_status"] = PRECHECK_MODEL_SOURCE_BINDING.get("binding_status", "<absent>")
    MODEL_DOWNLOAD_SUMMARY["binding_source"] = PRECHECK_MODEL_SOURCE_BINDING.get("binding_source", "<absent>")
    MODEL_DOWNLOAD_SUMMARY["binding_reason"] = PRECHECK_MODEL_SOURCE_BINDING.get("binding_reason", "<absent>")
    MODEL_DOWNLOAD_SUMMARY["model_source_binding"] = PRECHECK_MODEL_SOURCE_BINDING

record_precheck(
    "precheck 绑定配置路径",
    PW01_BOUND_CONFIG_PATH.exists(),
    str(PW01_BOUND_CONFIG_PATH),
)
record_precheck(
    "notebook 模型绑定已应用到 precheck 配置",
    PRECHECK_BINDING_APPLIED,
    json.dumps(PRECHECK_MODEL_SOURCE_BINDING, ensure_ascii=False, sort_keys=True)
    if PRECHECK_MODEL_SOURCE_BINDING is not None
    else "<absent>",
)

PW01_PREFLIGHT = detect_pw01_preflight(PW01_BOUND_CONFIG_PATH)
record_precheck(
    "PW01 preflight",
    PW01_PREFLIGHT["ok"],
    json.dumps(PW01_PREFLIGHT, ensure_ascii=False, sort_keys=True),
)
record_precheck(
    "CUDA 工具可用",
    PW01_PREFLIGHT["gpu_tool_available"],
    PW01_PREFLIGHT["nvidia_smi_path"],
)
record_precheck(
    "torch.cuda.is_available()",
    torch.cuda.is_available(),
    f"device_count={torch.cuda.device_count() if torch.cuda.is_available() else 0}",
)
record_precheck(
    "attestation env 已就绪",
    not PW01_PREFLIGHT["missing_attestation_env_vars"],
    ", ".join(PW01_PREFLIGHT["missing_attestation_env_vars"]) or "ok",
)
record_precheck(
    "attestation env 文件存在",
    str(ATTESTATION_BOOTSTRAP.get("status", "")) == "disabled"
    or (
        Path(str(ATTESTATION_BOOTSTRAP.get("attestation_env_path", ""))).exists()
        and Path(str(ATTESTATION_BOOTSTRAP.get("attestation_env_info_path", ""))).exists()
    ),
    json.dumps(
        {
            "attestation_env_path": ATTESTATION_BOOTSTRAP.get("attestation_env_path", "<absent>"),
            "attestation_env_info_path": ATTESTATION_BOOTSTRAP.get("attestation_env_info_path", "<absent>"),
        },
        ensure_ascii=False,
        sort_keys=True,
    ),
)
record_precheck(
    "InSPyReNet 权重存在",
    WEIGHT_PATH.exists() and WEIGHT_PATH.is_file(),
    f"path={WEIGHT_PATH}, persistent_path={PERSISTENT_WEIGHT_PATH}",
)

nvidia_smi_result = subprocess.run(
    ["nvidia-smi"],
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
nvidia_smi_output = (nvidia_smi_result.stdout or nvidia_smi_result.stderr).splitlines()
record_precheck(
    "nvidia-smi 命令返回 0",
    nvidia_smi_result.returncode == 0,
    " | ".join(nvidia_smi_output[:4]) if nvidia_smi_output else "<no_output>",
)

for module_name in [
    "yaml",
    "huggingface_hub",
    "diffusers",
    "transformers",
    "safetensors",
    "transparent_background",
    "torch",
]:
    try:
        __import__(module_name)
        record_precheck(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        record_precheck(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

hf_model_ok = False
hf_model_detail = "not_checked"
try:
    socket.setdefaulttimeout(15)
    model_info = HfApi().model_info(str(MODEL_IDENTITY["model_id"]))
    hf_model_ok = True
    hf_model_detail = f"accessible: {model_info.id}"
except Exception as exc:
    hf_model_ok = False
    hf_model_detail = f"{type(exc).__name__}: {exc}"
record_precheck("HF 模型可访问", hf_model_ok, hf_model_detail)

disk_usage = shutil.disk_usage(str(REPO_ROOT))
free_gb = disk_usage.free / (1024 ** 3)
record_precheck("磁盘剩余空间", free_gb >= 20.0, f"free={free_gb:.1f} GB，建议 >= 20 GB")

print_json(
    "pw01_precheck",
    {
        "family_id": FAMILY_ID,
        "sample_role": SAMPLE_ROLE,
        "sample_role_dirname": SAMPLE_ROLE_DIRNAME,
        "shard_index": SHARD_INDEX,
        "shard_count": SHARD_COUNT,
        "pw01_worker_count": PW01_WORKER_COUNT,
        "config_path": str(CONFIG_PATH),
        "local_hf_home": str(LOCAL_HF_HOME),
        "model_snapshot_path": str(MODEL_SNAPSHOT_PATH),
        "persistent_inspyrenet_path": str(PERSISTENT_WEIGHT_PATH),
        "repo_inspyrenet_path": str(WEIGHT_PATH),
        "snapshot_source": MODEL_CACHE_BOOTSTRAP.get("snapshot_source", "<absent>"),
        "cache_reuse_mode": MODEL_CACHE_BOOTSTRAP.get("cache_reuse_mode", "<absent>"),
        "precheck_bound_config_path": str(PRECHECK_BOUND_CONFIG_PATH),
        "bound_config_path": str(PW01_BOUND_CONFIG_PATH),
        "notebook_model_snapshot_binding_applied": PRECHECK_BINDING_APPLIED,
        "precheck_bound_config": {
            "model_snapshot_path": PRECHECK_BOUND_CFG.get("model_snapshot_path", "<absent>"),
            "model_source_binding": PRECHECK_MODEL_SOURCE_BINDING or "<absent>",
        },
        "items": PRECHECK_RESULTS,
        "pw01_preflight": PW01_PREFLIGHT,
    },
)

hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"PW01 precheck failed: {hard_fail}")

## 5) 执行 PW01 CLI

本单元执行单 shard 任务；若目标 shard 已完成，FORCE_RERUN=True 时会附加 --force-rerun。

In [ ]:
from datetime import datetime, timezone
import time

from paper_workflow.scripts.pw_local_runtime import archive_local_runtime_for_stage
from scripts.gpu_session_peak import build_gpu_peak_display_summary
from scripts.notebook_runtime_common import (
    build_repo_import_subprocess_env,
    build_stage_runtime_diagnostics_payload,
    build_stage_runtime_workload_summary,
    write_stage_runtime_diagnostics,
)

FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
SHARD_ROOT = FAMILY_ROOT / "source_shards" / SAMPLE_ROLE_DIRNAME / f"shard_{SHARD_INDEX:04d}"
PW01_SHARD_MANIFEST_PATH = SHARD_ROOT / "shard_manifest.json"
PW01_RUNTIME_DIAGNOSTICS_PATH = (
    FAMILY_ROOT / "runtime_state" / f"pw01_{SAMPLE_ROLE}_shard_{SHARD_INDEX:04d}_runtime_diagnostics.json"
 )
GPU_PEAK_SCRIPT_PATH = REPO_ROOT / "scripts" / "gpu_session_peak.py"
GPU_PEAK_SUMMARY_PATH = SHARD_ROOT / "artifacts" / "gpu_session_peak.json"
COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--family-id",
    FAMILY_ID,
    "--sample-role",
    SAMPLE_ROLE,
    "--shard-index",
    str(SHARD_INDEX),
    "--shard-count",
    str(SHARD_COUNT),
    "--pw01-worker-count",
    str(PW01_WORKER_COUNT),
    "--bound-config-path",
    str(PW01_BOUND_CONFIG_PATH),
]
if FORCE_RERUN:
    COMMAND.append("--force-rerun")
MONITORED_COMMAND = [
    sys.executable,
    str(GPU_PEAK_SCRIPT_PATH),
    "--output-json",
    str(GPU_PEAK_SUMMARY_PATH),
    "--label",
    NOTEBOOK_NAME,
    "--sample-interval-ms",
    "200",
    "--",
    *COMMAND,
]

NOTEBOOK_SUBPROCESS_ENV = build_repo_import_subprocess_env(repo_root=REPO_ROOT)
NOTEBOOK_SUBPROCESS_ENV["CEG_WM_MODEL_SNAPSHOT_PATH"] = str(MODEL_SNAPSHOT_PATH)
RUN_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()
RUN_STARTED_AT_MONOTONIC = time.perf_counter()
PW01_RESULT = subprocess.run(
    MONITORED_COMMAND,
    cwd=REPO_ROOT,
    env=NOTEBOOK_SUBPROCESS_ENV,
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
RUN_FINISHED_AT_UTC = datetime.now(timezone.utc).isoformat()
RUN_ELAPSED_SECONDS = time.perf_counter() - RUN_STARTED_AT_MONOTONIC

if PW01_RESULT.returncode != 0:
    expected_shard_root = FAMILY_ROOT / "source_shards" / SAMPLE_ROLE_DIRNAME / f"shard_{SHARD_INDEX:04d}"
    recent_worker_logs = []
    worker_root = expected_shard_root / "workers"
    if worker_root.exists():
        for log_path in sorted(
            worker_root.rglob("*.log"),
            key=lambda item: item.stat().st_mtime if item.exists() else 0,
            reverse=True,
        )[:6]:
            recent_worker_logs.append(
                {
                    "path": str(log_path),
                    "tail": log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:],
                }
            )
    failure_diagnostics = {
        "return_code": PW01_RESULT.returncode,
        "command": COMMAND,
        "monitored_command": MONITORED_COMMAND,
        "stdout_tail": PW01_RESULT.stdout.splitlines()[-40:],
        "stderr_tail": PW01_RESULT.stderr.splitlines()[-40:],
        "sample_role": SAMPLE_ROLE,
        "expected_shard_root": str(expected_shard_root),
        "gpu_session_peak_summary_path": str(GPU_PEAK_SUMMARY_PATH),
        "gpu_session_peak_summary": (
            json.loads(GPU_PEAK_SUMMARY_PATH.read_text(encoding="utf-8"))
            if GPU_PEAK_SUMMARY_PATH.exists() and GPU_PEAK_SUMMARY_PATH.is_file()
            else None
        ),
        "recent_worker_logs": recent_worker_logs,
    }
    print_json("pw01_failure_diagnostics", failure_diagnostics)
    raise RuntimeError(
        "PW01 failed; inspect pw01_failure_diagnostics for command output, GPU peak summary, and worker log tails"
    )

if not PW01_SHARD_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "PW01 shard manifest missing after successful subprocess execution: "
        f"shard_manifest_path={PW01_SHARD_MANIFEST_PATH} stdout={PW01_RESULT.stdout} stderr={PW01_RESULT.stderr}"
    )

PW01_SUMMARY = json.loads(PW01_SHARD_MANIFEST_PATH.read_text(encoding="utf-8"))
GPU_PEAK_SUMMARY = json.loads(GPU_PEAK_SUMMARY_PATH.read_text(encoding="utf-8"))
GPU_PEAK_NOTEBOOK_SUMMARY = build_gpu_peak_display_summary(GPU_PEAK_SUMMARY)
GPU_PEAK_NOTEBOOK_SUMMARY["summary_path"] = str(GPU_PEAK_SUMMARY_PATH)
print_json("pw01_summary", PW01_SUMMARY)
print_json("gpu_session_peak_summary", GPU_PEAK_NOTEBOOK_SUMMARY)

def runtime_count(value):
    return int(value) if isinstance(value, int) and not isinstance(value, bool) and value >= 0 else 0

PW01_EVENT_COUNT = runtime_count(PW01_SUMMARY.get("event_count"))
worker_rows = PW01_SUMMARY.get("workers", [])

if isinstance(worker_rows, list) and worker_rows:
    PW01_COMPLETED_EVENT_COUNT = sum(
        len(worker.get("completed_event_ids", []))
        for worker in worker_rows
        if isinstance(worker, dict) and worker.get("status") == "completed"
    )
else:
    PW01_COMPLETED_EVENT_COUNT = (
        PW01_EVENT_COUNT
        if PW01_SUMMARY.get("status") == "completed"
        else 0
    )

PW01_COMPLETED_EVENT_COUNT = min(PW01_EVENT_COUNT, runtime_count(PW01_COMPLETED_EVENT_COUNT))
PW01_FAILED_EVENT_COUNT = max(0, PW01_EVENT_COUNT - PW01_COMPLETED_EVENT_COUNT)
PW01_COUNT_SUMMARY = {
    "event_count": PW01_EVENT_COUNT,
    "completed_event_count": runtime_count(PW01_COMPLETED_EVENT_COUNT),
    "failed_event_count": runtime_count(PW01_FAILED_EVENT_COUNT),
    "source_shard_count": runtime_count(PW01_SUMMARY.get("source_shard_count")),
    "pw01_worker_count": runtime_count(PW01_SUMMARY.get("pw01_worker_count")),
}
PW01_RUNTIME_DIAGNOSTICS = build_stage_runtime_diagnostics_payload(
    stage_name="PW01_Source_Event_Shards",
    family_id=FAMILY_ID,
    expected_output_label="pw01_shard_manifest",
    expected_output_path=PW01_SHARD_MANIFEST_PATH,
    started_at_utc=RUN_STARTED_AT_UTC,
    finished_at_utc=RUN_FINISHED_AT_UTC,
    elapsed_seconds=RUN_ELAPSED_SECONDS,
    return_code=PW01_RESULT.returncode,
    stdout_text=PW01_RESULT.stdout,
    stderr_text=PW01_RESULT.stderr,
    count_summary=PW01_COUNT_SUMMARY,
    workload_summary=build_stage_runtime_workload_summary(
        unit_label="source_events",
        unit_count=PW01_COUNT_SUMMARY["event_count"],
        elapsed_seconds=RUN_ELAPSED_SECONDS,
    ),
    shard_index=SHARD_INDEX,
    sample_role=SAMPLE_ROLE,
    gpu_session_peak_path=GPU_PEAK_SUMMARY_PATH,
    monitor_status=(
        str(GPU_PEAK_SUMMARY.get("status"))
        if isinstance(GPU_PEAK_SUMMARY.get("status"), str)
        else None
    ),
)
write_stage_runtime_diagnostics(
    diagnostics_path=PW01_RUNTIME_DIAGNOSTICS_PATH,
    payload=PW01_RUNTIME_DIAGNOSTICS,
)
print_json("pw01_runtime_diagnostics", PW01_RUNTIME_DIAGNOSTICS)
if LOCAL_RUNTIME_ENABLED:
    archive_local_runtime_for_stage(
        stage_name="PW01_Source_Event_Shards",
        family_id=FAMILY_ID,
        local_project_root=LOCAL_PROJECT_ROOT,
        drive_bundle_root=DRIVE_BUNDLE_ROOT,
        sample_role=SAMPLE_ROLE,
        shard_index=SHARD_INDEX,
        shard_count=SHARD_COUNT,
        clean_after_archive=False,
    )

## 6) 回读 shard manifest

本单元验证 shard_manifest 已生成且状态为 completed。

In [ ]:
PW01_RESULT_SUMMARY = {
    "family_id": FAMILY_ID,
    "sample_role": SAMPLE_ROLE,
    "shard_index": SHARD_INDEX,
    "shard_count": SHARD_COUNT,
    "pw01_worker_count": PW01_WORKER_COUNT,
    "status": PW01_SUMMARY.get("status"),
    "model_snapshot_path": str(MODEL_SNAPSHOT_PATH),
    "persistent_inspyrenet_path": str(PERSISTENT_WEIGHT_PATH),
    "repo_inspyrenet_path": str(WEIGHT_PATH),
    "gpu_session_peak_path": str(GPU_PEAK_SUMMARY_PATH),
    "snapshot_source": MODEL_CACHE_BOOTSTRAP.get("snapshot_source", "<absent>"),
    "cache_reuse_mode": MODEL_CACHE_BOOTSTRAP.get("cache_reuse_mode", "<absent>"),
}
print_json("pw01_result_summary", PW01_RESULT_SUMMARY)

## 7) 如何扩展并行（显式说明）

扩展规则：
1. 先在 PW00 固定好 source_shard_count，并优先完成 formal 主流程的 positive_source 与 clean_negative。
2. 多个执行器按 shard_index 分片并行执行 PW01。
3. 每个 shard 只写入与当前 sample_role 对应的 source_shards/positive/shard_xxxx、source_shards/negative/shard_xxxx 或 source_shards/control_negative/shard_xxxx，不会互相覆盖。
4. planner_conditioned_control_negative 仅在需要 diagnostic pool 时按需补跑。
5. 单个 shard 内可通过 PW01_WORKER_COUNT 控制是单路还是双 worker。
6. 双 worker 的独立日志与结果只写入当前 shard 的 workers/worker_XX。
7. 仅在需要覆盖历史产物时使用 --force-rerun。

In [ ]:
print("[PW01 后续并行运行说明]")
print(f"\n当前 FAMILY_ID = {FAMILY_ID}")
print(f"当前 sample_role：{SAMPLE_ROLE}")
print(f"\n当前 shard 设置：SHARD_COUNT = {SHARD_COUNT}")
print(f"当前 notebook 只运行：{SAMPLE_ROLE} shard {SHARD_INDEX}")
print(f"该 sample_role 需要完成：shard 0-{SHARD_COUNT - 1}")

print("PW02 运行条件：positive_source 与 clean_negative 的全部 shard 均完成后，运行 PW02 一次。")

## 8) 双路有效性说明

- PW01_WORKER_COUNT = 2 只表示允许在单个 PW01 shard 内启动两个本地 event worker。
- 是否真的形成“双路有效”，取决于当前 shard 实际分到的 event 数是否至少为 2。
- 当前总事件数 = prompt_count × seed_count。
- 当前 shard 分片规则 = event_index % source_shard_count。
- 若希望两个隔离会话都形成有效双路，至少需要满足 total_event_count >= 2 × source_shard_count。
- 每个 worker 只处理当前 shard 内按本地顺序编号切分后的 event 子集；正式 event 落盘仍位于当前 shard 的 events/、records/、artifacts/ 下，worker 自身日志与结果位于 workers/worker_XX/。